<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Test a model or a prompt online

**[Test a model or a prompt online](https://learning.nextia-ai.com/courses/experiments/m08/test-a-model-or-a-prompt-online/)** · Experiments and Causal Impact: Did Our Change Work? · Module 8, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** grade two prompts of Larkfield's support assistant offline on recorded answers, then analyse their online test per ticket, with the tickets grouped by customer, the randomisation unit.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the assistant's answers were recorded once and are replayed. |
| **You should know** | Ratio metrics, from [Choose the metrics](https://learning.nextia-ai.com/courses/experiments/m02/choose-the-metrics/); offline evaluation, from the [evaluation course](https://learning.nextia-ai.com/courses/evals/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m08/test-a-model-or-a-prompt-online/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M08-L01-test-a-model-or-a-prompt-online/test-a-model-or-a-prompt-online-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 3 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "support_answers.jsonl": "a0e97d52f9543ecf1dc487216d44010a979e782342ba388b2e5fba21d24af55b",
    "prompt/tickets.csv.gz": "b0e62735d47001c6381d4851382bbd8f4ed28982a2cb33b03cfbd98b84699b1d",
    "prompt/truth.json": "cd4f1652ab7a06bf4627b72184408d1d3524ffa548595a7b4016f7698926e45b"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The recorded answers

120 support questions, each answered twice by a language model: with today's prompt **A** and with the new prompt **B**. The answers were recorded once, so everyone sees the same ones.

In [ ]:
answers = pd.read_json("data/support_answers.jsonl", lines=True)
print(answers.shape)
for _, r in answers[answers.id == "change_order-01"].iterrows():
    print(f"[{r.prompt}] {r.answer}\n")

> **Check.** You should see `(240, 10)` and the two answers to "Can I change my order?".

## 2. Offline: grade every answer with simple rules

A rule says what a good answer is: it gives the policy fact (a pattern), and it hands over to a person only when the policy says so.

In [ ]:
import re
HANDOFF = re.compile(r"connect(ing)? you (with|to)|support agent|transfer(ring)? you|human agent|"
                     r"(a|our) (team member|specialist|person) (will|can)|escalat", re.I)

def grade(r):
    a = r.answer.replace("\u2019", "'")
    handoff = bool(HANDOFF.search(a))
    if r.must_hand_off:
        return pd.Series({"correct": int(handoff), "handoff": int(handoff)})
    return pd.Series({"correct": int(bool(re.search(r.fact, a, re.I)) and not handoff), "handoff": int(handoff)})

answers[["correct", "handoff"]] = answers.apply(grade, axis=1)
answers["words"] = answers.answer.str.split().str.len()
answers.groupby("prompt")[["correct", "handoff", "words", "tokens_out"]].mean()

> **Check.** You should see prompt A correct 0.9083 and prompt B correct 0.9917; B hands off less and writes about half as many words.

Offline, B is better. Offline grades say nothing about how customers react: do they get their answer, ask for a person, complain? That needs an online test.

## 3. Online: per ticket, grouped by customer

For 14 days, a coin flip per **customer** chose the prompt. A customer can open several tickets, and their tickets are not independent. So we add up per customer and use the ratio of sums (the delta method of Module 2, lesson 2).

In [ ]:
tickets = pd.read_csv("data/prompt/tickets.csv.gz")
per = tickets.groupby(["customer_id", "variant"]).agg(n=("ticket_id", "size"), resolved=("resolved", "sum"),
                                                      handoff=("handoff", "sum"), cost=("cost", "sum")).reset_index()

def ratio_diff(col):
    out = []
    for v in ("A", "B"):
        g = per[per.variant == v]
        r = g[col].sum() / g.n.sum()
        cov = np.cov(g[col], g.n)
        var = (cov[0, 0] - 2 * r * cov[0, 1] + r**2 * cov[1, 1]) / (g.n.mean() ** 2 * len(g))
        out.append((r, var))
    d, se = out[1][0] - out[0][0], np.sqrt(out[0][1] + out[1][1])
    return out[0][0], out[1][0], d, d - 1.96 * se, d + 1.96 * se

print(f"{len(tickets):,} tickets from {len(per):,} customers")
for col in ("resolved", "handoff"):
    a, b, d, lo, hi = ratio_diff(col)
    print(f"{col:<9} A {a:.4f}  B {b:.4f}  B − A {d:+.4f} ({lo:+.4f} to {hi:+.4f})")

> **Check.** You should see `21,357 tickets from 9,000 customers`, and resolved A 0.7007, B 0.7982.

## 4. Your turn: the cost per ticket

The cost of a ticket is the model's tokens plus 6.50 dollars of an agent's time when the ticket is handed over (an assumption).

> **Your turn.** Use `ratio_diff("cost")` to compute the difference B − A in cost per ticket. Put the difference, rounded to 4 decimals, in `cost_diff`. Then compare it with `truth("prompt")["cost"]["effect"]`.

In [ ]:
cost_diff = ...  # B - A in dollars per ticket, rounded to 4 decimals

In [ ]:
expect('cost_diff', cost_diff, -0.6415)

The new prompt saves about 0.64 dollars a ticket, mostly through fewer hand-offs. The model's own cost is less than a tenth of a cent a ticket: the agent's time is the cost that matters.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m08/test-a-model-or-a-prompt-online/) has the guardrails (complaints, unsafe answers) and the knowledge checks that count toward your certificate. In the project: `xp prompt --offline`, then `sim run --scenario prompt` and `xp prompt`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Bandits](https://learning.nextia-ai.com/courses/experiments/m08/bandits/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m08/test-a-model-or-a-prompt-online/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.